# 03 - Particiones de Spark y particiones de almacenamiento

## Conexión con la teoría

Veremos el paralelismo potencial, `repartition`, `coalesce`, la relación entre particiones y
archivos de salida, el problema de los small files y la diferencia con `partitionBy`.

In [ ]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Medir la distribución de filas por partición

In [ ]:
records = (
    spark.range(0, 50_000, numPartitions=4)
    .withColumn("year", F.lit(2026))
    .withColumn("month", ((F.col("id") % 3) + 1).cast("int"))
    .withColumn("category", F.concat(F.lit("cat_"), F.col("id") % 10))
)

def rows_per_partition(df):
    return (
        df.withColumn("partition_id", F.spark_partition_id())
        .groupBy("partition_id")
        .count()
        .orderBy("partition_id")
    )

print("Particiones:", records.rdd.getNumPartitions())
rows_per_partition(records).show()

## 2. `repartition` redistribuye; `coalesce` normalmente colapsa

In [ ]:
repartitioned = records.repartition(8)
coalesced = repartitioned.coalesce(2)

print("Original:", records.rdd.getNumPartitions())
print("Tras repartition(8):", repartitioned.rdd.getNumPartitions())
print("Tras coalesce(2):", coalesced.rdd.getNumPartitions())

repartitioned.explain(mode="formatted")
coalesced.explain(mode="formatted")

Busca `Exchange RoundRobinPartitioning` en `repartition`. `coalesce` puede reducir particiones
sin un shuffle completo, pero una reducción excesiva también reduce el paralelismo.

## 3. Una partición de ejecución suele generar un archivo

In [ ]:
def count_part_files(path: Path) -> int:
    return len(list(path.rglob("part-*")))

many_files_path = DATA_DIR / "many_files"
few_files_path = DATA_DIR / "few_files"

repartitioned.write.mode("overwrite").parquet(str(many_files_path))
coalesced.write.mode("overwrite").parquet(str(few_files_path))

print("Archivos con 8 particiones:", count_part_files(many_files_path))
print("Archivos con 2 particiones:", count_part_files(few_files_path))

No existe una regla universal de tamaño. El objetivo es evitar miles de archivos diminutos sin
crear archivos tan grandes que limiten el paralelismo o dificulten su manejo.

## 4. `partitionBy` organiza el almacenamiento por columnas

In [ ]:
storage_partitioned_path = DATA_DIR / "partitioned_by_month"
(
    records
    .coalesce(3)
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(storage_partitioned_path))
)

for directory in sorted(p for p in storage_partitioned_path.rglob("*") if p.is_dir()):
    print(directory.relative_to(storage_partitioned_path))

Las carpetas `year=.../month=...` son particiones de almacenamiento. No equivalen uno a uno a
las particiones que Spark usa para ejecutar tasks.

## 5. Partition pruning

In [ ]:
stored = spark.read.parquet(str(storage_partitioned_path))
only_february = stored.filter((F.col("year") == 2026) & (F.col("month") == 2))
only_february.explain(mode="formatted")
print("Filas de febrero:", only_february.count())

Busca `PartitionFilters` en el scan. Spark puede descartar directorios completos antes de leer
los archivos que contienen.

## Preguntas de cierre

1. ¿Por qué ocho particiones no implican ocho tasks simultáneas?
2. ¿Qué coste introduce `repartition`?
3. ¿Por qué `coalesce(1)` suele ser un antipatrón al escribir grandes resultados?
4. ¿En qué se diferencia una partición de Spark de una partición `year/month` del data lake?

In [ ]:
spark.stop()